## BERT Sentiment Explainability

Why did BERT classify a review as positive, negative, or neutral? Which aspects of the product (or the service) contributed to the score and classification?  

To answer these questions, we will use model explainability techniques to analyze the model's predictions and understand the factors influencing its decisions.

In [16]:
import json
import re
import time
from pathlib import Path
from collections import defaultdict
 
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers_interpret import SequenceClassificationExplainer

In [5]:
MODEL_DIR = Path('../src/models/bert-reviews-tuned')
CLASS_ORDER = ["negative", "neutral", "positive"]
DATA_DIR  = Path('../data/processed')
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
 
cls_explainer = SequenceClassificationExplainer(model, tokenizer)
 
N_STEPS = 25 

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [17]:
def truncate_to_max_tokens(text, max_tokens=512):
    '''truncates the text not to conflict with BERT's limit'''
    token_ids = tokenizer.encode(text, max_length=max_tokens, truncation=True)
    return tokenizer.decode(token_ids, skip_special_tokens=True)

In [18]:
df_test = pd.read_csv(DATA_DIR / 'test_results.csv')
df_test["text"] = df_test["text"].apply(truncate_to_max_tokens)
 
print(df_test.shape)

(300, 3)


In [19]:
STOPWORDS = {"the", "a", "an", "is", "it", "this", "that", "and", "to", "of", "in", "i", "for", "was", "with"}
 
def merge_subwords(word_attributions):
    merged = []
    current_word, current_scores = "", []
    for token, score in word_attributions:
        if token in ("[CLS]", "[SEP]", "[PAD]"):
            continue
        if token.startswith("##"):
            current_word += token[2:]
            current_scores.append(score)
        else:
            if current_word:
                merged.append((current_word, sum(current_scores) / len(current_scores)))
            current_word, current_scores = token, [score]
    if current_word:
        merged.append((current_word, sum(current_scores) / len(current_scores)))
    return merged
 
def clean_word(w):
    w = w.lower().strip()
    w = re.sub(r"[^a-zà-ù]", "", w)
    return w

In [20]:
ASPECT_KEYWORDS = {
    "product_quality": {"taste", "flavor", "fresh", "stale", "quality", "delicious",
                         "texture", "smell", "ingredients", "good", "bad", "love", "great"},
    "shipping": {"shipping", "delivery", "arrived", "box", "package", "packaging", "shipped"},
    "price": {"price", "expensive", "cheap", "cost", "value", "worth"},
    "service": {"seller", "amazon", "customer", "service", "refund", "return"},
}

def aspect_of(word):
    for aspect, keywords in ASPECT_KEYWORDS.items():
        if word in keywords:
            return aspect
    return "other"

In [ ]:
agg_by_class = defaultdict(lambda: defaultdict(lambda: [0.0, 0]))  # classe -> parola -> [score_sum, count]
review_rows = []
 
start = time.time()
 
for idx, row in df_test.reset_index(drop=True).iterrows():
    word_attributions = cls_explainer(row["text"], n_steps=N_STEPS)
    merged = merge_subwords(word_attributions)
 
    aspect_weight = defaultdict(float)
 
    for word, score in merged:
        w = clean_word(word)
        if not w or w in STOPWORDS or len(w) < 3:
            continue
 
        agg_by_class[row["pred_label"]][w][0] += abs(score)
        agg_by_class[row["pred_label"]][w][1] += 1
 
        aspect_weight[aspect_of(w)] += abs(score)
 
    total = sum(aspect_weight.values()) or 1.0
    distribution = {a: w / total for a, w in aspect_weight.items()}
    predominant = max(distribution, key=distribution.get) if distribution else "other"
 
    review_rows.append({
        "predominant_aspect": predominant,
        "aspect_distribution": distribution,
    })
 
    if idx % 50 == 0:
        print(f"{idx}/{len(df_test)} - {time.time() - start:.0f}s")
 
aspect_df = pd.DataFrame(review_rows)
df_test = df_test.reset_index(drop=True).join(aspect_df)
df_test["is_correct"] = df_test["label"] == df_test["pred_label"]
 
print(f"Tempo totale: {(time.time() - start) / 60:.1f} min")

0/300 - 12s


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained('../src/models/bert-reviews-tuned', num_labels=3)
tokenizer = AutoTokenizer.from_pretrained('../src/models/bert-reviews-tuned')

from transformers_interpret import SequenceClassificationExplainer

cls_explainer = SequenceClassificationExplainer(
    model,
    tokenizer,
)
word_attributions = cls_explainer("This stuff is awful, cheap leaves and just not good. I've made better tea drying blackberry leaves! Stick to Liptons or Red Rose!!")
print(f'Predicted label {cls_explainer.predicted_class_index} aka {cls_explainer.predicted_class_name} label')
_ = cls_explainer.visualize()

In [ ]:
import shap
import transformers

# Carica il modello e il tokenizer
model = transformers.AutoModelForSequenceClassification.from_pretrained("../src/models/bert-reviews-tuned", num_labels=3)
tokenizer = transformers.AutoTokenizer.from_pretrained("../src/models/bert-reviews-tuned")

# Crea una pipeline di Hugging Face
pred_pipeline = transformers.pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, return_all_scores=True)

# Crea l'explainer di SHAP
explainer = shap.Explainer(pred_pipeline)

# Esegui l'analisi su un esempio
text = ["This stuff is awful, cheap leaves and just not good. I've made better tea drying blackberry leaves! Stick to Liptons or Red Rose!!"]
shap_values = explainer(text)

# Visualizza
shap.plots.text(shap_values[0])